# A2.2 N-Gram Language Modeling

Uses `brown_100.txt` (100 sentences from the Brown Corpus). Writes all outputs to `./results/`.

In [ ]:
import os
import re
import math
import random
from collections import defaultdict

os.makedirs('./results', exist_ok=True)

with open('./brown_100.txt', 'r') as f:
    corpus = f.read()


def save_results(items, filename):
    # Write each ngram and its probability on its own line, like: the jury | 0.083
    lines = []
    for ngram, prob in items:
        lines.append(' '.join(ngram) + ' | ' + str(prob))
    with open(f'./results/{filename}.txt', 'w') as f:
        f.write('\n'.join(lines))


print('corpus length:', len(corpus), 'chars')

## NGramModel class

Four steps: tokenize, build n-grams, count them, compute probabilities (with optional alpha smoothing).

In [ ]:
class NGramModel:
    def __init__(self, text, n, alpha=0.0):
        self.text = text
        self.n = n
        self.alpha = alpha
        self.ngrams = {}
        self.probabilities = {}
        self.vocab = set()
        self.sentences = []
        self.context_counts = {}
        self.total = 0

    def tokenize(self):
        text = self.text.lower()

        # The corpus already marks sentences with <s> ... </s>, so split on </s>
        parts = text.split('</s>')

        self.sentences = []
        tokens = []
        for part in parts:
            part = part.replace('<s>', '')
            # Keep letters, digits, and apostrophes (handles atlanta's, o'clock)
            words = re.findall(r"[a-z0-9']+", part)
            # Drop tokens that are only apostrophes (the '' quote marker)
            words = [w for w in words if any(c.isalnum() for c in w)]
            if len(words) == 0:
                continue
            sent = ['<s>'] + words + ['</s>']
            self.sentences.append(sent)
            tokens.extend(sent)

        self.vocab = set(t for t in tokens if t != '<s>' and t != '</s>')
        return tokens

    def generate_ngrams(self, tokens):
        # Sliding window inside each sentence so n-grams never cross sentence boundaries
        result = []
        for sent in self.sentences:
            if len(sent) < self.n:
                continue
            for i in range(len(sent) - self.n + 1):
                result.append(tuple(sent[i:i + self.n]))
        self.ngrams = result
        return result

    def count_frequencies(self):
        counts = defaultdict(int)
        for ng in self.ngrams:
            counts[ng] += 1
        self.ngrams = dict(counts)

    def calculate_probabilities(self):
        self.probabilities = {}
        V = len(self.vocab)
        a = self.alpha

        if self.n == 1:
            # Unigram: P(w) = (count(w) + a) / (total + a * V)
            total = sum(self.ngrams.values())
            self.total = total
            for ng, c in self.ngrams.items():
                self.probabilities[ng] = (c + a) / (total + a * V)
        else:
            # Higher-order: P(w_n | prefix) = (count + a) / (prefix_count + a * V)
            contexts = defaultdict(int)
            for ng, c in self.ngrams.items():
                contexts[ng[:-1]] += c
            self.context_counts = dict(contexts)
            for ng, c in self.ngrams.items():
                prefix = ng[:-1]
                self.probabilities[ng] = (c + a) / (contexts[prefix] + a * V)

    def most_frequent_ngrams(self, top_n=10):
        sorted_ngrams = sorted(self.ngrams.items(), key=lambda x: x[1], reverse=True)
        top = sorted_ngrams[:top_n]
        return [(ng, self.probabilities[ng]) for ng, c in top]


# Quick sanity check on a tiny sentence
demo = NGramModel('This is a simple example to demonstrate how n-grams work.', 2)
t = demo.tokenize()
demo.generate_ngrams(t)
demo.count_frequencies()
demo.calculate_probabilities()
print(demo.most_frequent_ngrams(5))

### Unigrams

In [ ]:
model = NGramModel(corpus, 1)
tokens = model.tokenize()
model.generate_ngrams(tokens)
model.count_frequencies()
model.calculate_probabilities()
save_results(model.most_frequent_ngrams(10), 'unigrams')
print(model.most_frequent_ngrams(10))

### Bigrams

In [ ]:
model = NGramModel(corpus, 2)
tokens = model.tokenize()
model.generate_ngrams(tokens)
model.count_frequencies()
model.calculate_probabilities()
save_results(model.most_frequent_ngrams(10), 'bigrams')
print(model.most_frequent_ngrams(10))

### Trigrams

In [ ]:
model = NGramModel(corpus, 3)
tokens = model.tokenize()
model.generate_ngrams(tokens)
model.count_frequencies()
model.calculate_probabilities()
save_results(model.most_frequent_ngrams(10), 'trigrams')
print(model.most_frequent_ngrams(10))

### Smoothing

Setting `alpha > 0` gives unseen n-grams a non-zero probability.

In [ ]:
# Bigrams with Laplace (add-one) smoothing
model = NGramModel(corpus, 2, alpha=1.0)
tokens = model.tokenize()
model.generate_ngrams(tokens)
model.count_frequencies()
model.calculate_probabilities()
save_results(model.most_frequent_ngrams(10), 'bigrams_smoothed')
print(model.most_frequent_ngrams(10))

### Text generation

Pick the next word by weighted random sampling over the model's probabilities. A fixed seed makes it reproducible.

In [ ]:
def generate_text(model, n, prompt, max_length=50, seed=42):
    rng = random.Random(seed)

    # Group candidates by their context so lookups are fast
    by_context = defaultdict(list)
    for ng, p in model.probabilities.items():
        by_context[ng[:-1]].append((ng[-1], p))

    tokens = prompt.lower().split()
    if len(tokens) == 0:
        tokens = ['<s>']

    while len(tokens) < max_length:
        context = tuple(tokens[-(n - 1):]) if n > 1 else ()
        candidates = by_context.get(context, [])
        if len(candidates) == 0:
            break

        # Weighted random sampling (not argmax, otherwise the output gets stuck in a loop)
        words = [w for w, _ in candidates]
        weights = [p for _, p in candidates]
        next_word = rng.choices(words, weights=weights, k=1)[0]
        if next_word == '</s>':
            break
        tokens.append(next_word)

    return ' '.join(tokens)


model = NGramModel(corpus, 2, alpha=1.0)
tokens = model.tokenize()
model.generate_ngrams(tokens)
model.count_frequencies()
model.calculate_probabilities()

output = generate_text(model, 2, 'the jury')
print(output)
with open('./results/generated_bigrams.txt', 'w') as f:
    f.write(output)

### Perplexity

Perplexity = `exp(- average log probability per token)`. Lower is better.
Split the corpus 90/10 on sentence boundaries, train on 90%, measure on the held-out 10%.

In [ ]:
def perplexity(model, test_sentences):
    V = len(model.vocab)
    a = model.alpha
    log_sum = 0.0
    count = 0

    for sent in test_sentences:
        if len(sent) < model.n:
            continue
        for i in range(len(sent) - model.n + 1):
            ng = tuple(sent[i:i + model.n])
            if ng in model.probabilities:
                p = model.probabilities[ng]
            else:
                # Unseen n-gram: fall back to the smoothing formula (numerator = alpha)
                if model.n == 1:
                    p = a / (model.total + a * V) if a > 0 else 0.0
                else:
                    ctx_count = model.context_counts.get(ng[:-1], 0)
                    denom = ctx_count + a * V
                    p = a / denom if denom > 0 else 0.0
            if p <= 0:
                return float('inf')
            log_sum += math.log(p)
            count += 1

    if count == 0:
        return float('inf')
    return math.exp(-log_sum / count)


# Build one string per sentence (each wrapped in <s> ... </s>), shuffle, then split 90/10
sentences = []
for piece in corpus.split('</s>'):
    piece = piece.strip()
    if piece:
        sentences.append(piece + ' </s>')

rng = random.Random(7)
rng.shuffle(sentences)
cut = int(0.9 * len(sentences))
train_text = ' '.join(sentences[:cut])
test_text = ' '.join(sentences[cut:])
print('train:', cut, 'sentences   test:', len(sentences) - cut, 'sentences')

# Tokenize the test set once
probe = NGramModel(test_text, 1)
probe.tokenize()
test_sentences = probe.sentences

results = []
for n in [1, 2, 3]:
    for a in [0.01, 0.1, 1.0, 5.0]:
        m = NGramModel(train_text, n, alpha=a)
        t = m.tokenize()
        m.generate_ngrams(t)
        m.count_frequencies()
        m.calculate_probabilities()
        ppl = perplexity(m, test_sentences)
        results.append((n, a, ppl))
        print(f'n={n}  alpha={a:<5}  perplexity={ppl:.2f}')

print('\nBest per n:')
for n in [1, 2, 3]:
    best = min((r for r in results if r[0] == n), key=lambda r: r[2])
    print(f'  n={best[0]}  alpha={best[1]}  perplexity={best[2]:.2f}')

### How alpha changes top-bigram probabilities

In [ ]:
# As alpha grows, probability mass spreads across the vocabulary, so top bigrams shrink
target = ('the', 'jury')
for a in [0.0, 0.01, 0.1, 1.0, 5.0]:
    m = NGramModel(corpus, 2, alpha=a)
    t = m.tokenize()
    m.generate_ngrams(t)
    m.count_frequencies()
    m.calculate_probabilities()
    p = m.probabilities.get(target, 0.0)
    print(f'alpha={a:<5}  P(the jury)={p:.6f}  top-3={m.most_frequent_ngrams(3)}')